# REDUCE and EXTRACT fixed slits for large numbers of NIRSpec datasets (prism and gratings)
jrigby. Original version from B. Welch  
Feb 2026:  Modified to run on multiple processors, which is a lot faster (but sometimes crashes the computer).
Followed [this page, example 2.](https://jwst-pipeline.readthedocs.io/en/1.20.0/jwst/user_documentation/running_pipeline_python.html#multiprocessing)
Various versions of this notebook reduced the fixed slit data for the He I interstellar background paper (frozen in a zenodo), Rosalia's prism zodiacal background paper, the JWST backgrounds paper II, and it keeps going.  Nov 2026

Upgraded to pipeline v3.0.0. Though documentation suggests running clean flicker noise at the Detector1 stage, this produced lots of artifacts when run lights-out.  (Some of those I've fixed, in my patch of the v3.0.0 jwst pipeline.)   Even w patching, it's much faster to separate out clean flicker noise from the v slow Detector1 stage.  After testing, I settled on this workflow:
- Run Detector1 with clean_flicker_noise OFF, to make a RATE file
- Make a custom mask (from the RATE file + MSA metadata file, or IFU slices, but altered by the actual illumination in the RATE file)
- Run Spec2 w patched pipeline with clean_flicker_noise ON to clean full detector, before cropping to fixed slits

RATE comparison in L2a: `*_rate.fits` is before cleaning; `*_rate_after_cleaning.fits` is after Stage 2 cleaning, before subsequent calibration. The cleaned image is saved without another ramp fit. Cleaning-step and fitted-noise diagnostic saves are disabled.

## Next time I reduce all this, prob in 2027, I should simplify the dir/ structure from DATE-OBS/VISITID/ to just VISITID/   It will keep visits from breaking at midnight, and by keeping more files in a dir, will make multiprocessing run faster.

In [1]:
import os
# 1) where the jwst pipeline config files are located
home = "/Users/jrrigby1/Ref_files/"

# Gemini suggested this, to avoid CPU collisions.  Keeps processes assigned to 1 core to that core. Try it.
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS']  = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# STScI helpdesk says these os commands need to come BEFORE jwst pipeline packages are imported
os.environ["CRDS_PATH"] = home + "crds_cache/jwst_ops"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
#os.environ["CRDS_CONTEXT"] =   'jwst_1464.pmap'     # for  pipeline v1.20.2, Fall 2025
#os.environ["CRDS_CONTEXT"] =   'jwst_1535.pmap'      # For pipeline v2.0.1 , Spring 2026
mycontext = 'jwst_1584.pmap'
os.environ["CRDS_CONTEXT"] =  mycontext   # v3.0.0

In [2]:
print("Computer has this many CPUs", int(os.cpu_count()))
max_filesize_multiprocess = 0.8E9  # If filesizes are >~1 GB, multiprocessing will crash the computer

Computer has this many CPUs 14


In [3]:
# Avoid re-running parts that finished  *** MODIFY AS NEEDED
#pipev = '2.0.1'  # Which version of the pipeline are we running
pipev = '3.0.0'
UNDO_EDITRAW = False
RUN_LEVEL0   = False
RUN_LEVEL1   = False
RUN_MASKS    = False
RUN_LEVEL2   = True
RUN_LEVEL3   = True
RUN_EXTRACT  = True

In [4]:
import numpy as np
import glob
from multiprocessing.pool import Pool
from os.path import basename, dirname, normpath
import matplotlib.pyplot as plt
import pandas
import re
from pathlib import Path
import sys
import importlib
from nirspec_ninja.pipeline.associations import writel3asn  # Tools from Brian Welch, David Law to make associations
from nirspec_ninja.fixed_slit.extraction import wrap_median_combine_level3_nirspecFS  # special median combine and extraction for FS backgrounds
from nirspec_ninja.pipeline.multiprocessing import predownload_crds
from nirspec_ninja.pipeline.multiprocessing import max_filesize
from nirspec_ninja.archive.download_msa_metadata import download_metadata # custom helper script to download MSA file
from nirspec_ninja.pipeline import notebook_workers 
importlib.reload(notebook_workers) 
from nirspec_ninja.pipeline.notebook_workers import run_jwst_det1, make_rate_custom_mask, run_jwst_spec2, run_jwst_spec3
from nirspec_ninja.pipeline.notebook_workers import run_extract, summarize_extract
from collections import defaultdict
from time import perf_counter
from datetime import datetime
import subprocess
from crds.core import rmap
plt.rcParams["figure.figsize"] = (9,4)
import warnings
warnings.filterwarnings('ignore') 

In [5]:
import json
from astropy.io import fits
from astropy.utils.data import download_file
import astropy.units as u
from astropy import wcs
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import matplotlib as mpl

import jwst
 
from jwst import datamodels

# association file utilities
from jwst.associations import asn_from_list as afl # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base # Definition of a Lvl3 association file

In [6]:
# This had better be what is expected, or STOP (1.20.2, or 2.0X)
if jwst.__version__ != pipev:
    raise Exception("ERROR, pipeline version", jwst.__version__, 'is not the expectation, pipev')
else : print('Good, pipeline version was as expected:', jwst.__version__)    

Good, pipeline version was as expected: 3.0.0


In [7]:
def poll_spectrum_at_wavelength(wave, spectrum, atwave=3.0):  # find the value of the spectrum at wavelength atwave
    idx = (np.abs(wave - atwave)).argmin()
    return spectrum[idx]

In [8]:
def sort_out_paths(thisdir, reduced_datadir, pipev='1.20.2'):  # Used several times, so generalize this 
    targname = thisdir.split('/')[-2]   # this is targname for pipev=1.20.2, and visitID for pipev=2
    input_path = thisdir  
    if pipev=='1.20.2':
        epochname =   basename(re.sub(r'\/$', '', thisdir))
        output_path = reduced_datadir + targname + '/' + epochname + '/'     # where to write result 
    elif ((pipev == '2.0.1') or (pipev == '3.0.0')):
        epochname = thisdir.split('/')[-3]   
        targname =  thisdir.split('/')[-2]   # this is visitID for pipev=2
        output_path = reduced_datadir + epochname + '/' + targname + '/' 
    return(epochname, targname, input_path, output_path)

In [9]:
#leggos_in =  '/Volumes/Rawdata/Raw_NIRSpecFS_for_PSF_measurement/G395M/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpecFS_reduced_PSF_subsamp_x4/G395M/'
leggos_in  = '/Volumes/Rawdata/RAW_prism_extra/'
leggos_out = '/Volumes/Rawdata/REDUCED_prism_extra/'
#leggos_in  = '/Volumes/Rawdata/RAW_prism/'
#leggos_out = '/Volumes/Rawdata/Reduced_prism/'
#leggos_in  = '/Volumes/Rawdata/Raw_MSAshorts/'
#leggos_out = '/Volumes/Rawdata/Reduced_MSAshorts/'
epochdirs = glob.glob(leggos_in + '*/*/' )

In [10]:
# Select which epochs to reduce.  All, or subset.  ***** MODIFY THIS *****
#dirs_to_handle = [x for x in epochdirs if 'VID01254006001' in x]
#dirs_to_handle = [leggos_in + x for x in missingL3]
dirs_to_handle = epochdirs
#dirs_to_handle

In [11]:
### Rerun the prisms that failed
#rerun_these = ['VID01128002001',]
#dirs_to_handle = [a for a in epochdirs if any(b in a for b in rerun_these)]
#dirs_to_handle

In [12]:
# Organize the output folders
folder_L2a = 'L2a/'
folder_L2b = 'L2b_pipeclean/'
folder_L3  = 'L3_destripe/'
folder_extracted = 'L3_extracted/'
output_folders = [folder_L2a, folder_L2b, folder_L3, folder_extracted]

In [13]:
# Make some dictionaries to hold files
raw_files = {}
l2a_files = {}
allmsa_calfiles = {}
allmsa_asnfile = {}

In [14]:
if RUN_LEVEL0 :   # prep the directories we'll need
    for ii, thisdir in enumerate(dirs_to_handle) :   
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        if os.path.exists(output_path) == False: # if folder doesn't exist
            print('   Creating folder ' + output_path)
            if pipev =='1.20.2':
                os.system('mkdir '  + leggos_out + '/' + targname) 
                os.system('mkdir '  + output_path) # creates the folder
            elif ((pipev == '2.0.1') or (pipev == '3.0.0')):                
                os.system('mkdir '  + leggos_out + '/' + epochname) 
                os.system('mkdir '  + output_path) # creates the folder
        
        for folder in output_folders :       #Make output folders if they don't already exist
            if os.path.exists(output_path + folder) == False: # if folder doesn't exist
                #print('   Creating folder ' + output_path + folder)
                os.system('mkdir ' + output_path + folder) # creates the folder

In [15]:
# get the MSA operability files needed for making masks for clean flicker noise
msa_mapping = rmap.get_cached_mapping(os.environ["CRDS_CONTEXT"]).get_imap("nirspec").get_rmap("msaoper")
print("Syncing MSA operability references from:", msa_mapping.basename)
subprocess.run([sys.executable, "-m", "crds.sync", "--contexts", msa_mapping.basename,
                "--fetch-references", "--check-files", "--check-sha1sum", "--repair-files"], check=True)

Syncing MSA operability references from: jwst_nirspec_msaoper_0019.rmap


CRDS - INFO -  Removing all context pickles.  Use --save-pickles to recreate for specified contexts.
CRDS - ERROR -  (FATAL) Failed connecting to CRDS server at CRDS_SERVER_URL = 'https://jwst-crds.stsci.edu' :: Required server connection unavailable.


CalledProcessError: Command '['/Users/jrrigby1/miniforge3/envs/JWSTDP-3.0.0-nsclean-patched/bin/python3.14', '-m', 'crds.sync', '--contexts', 'jwst_nirspec_msaoper_0019.rmap', '--fetch-references', '--check-files', '--check-sha1sum', '--repair-files']' returned non-zero exit status 255.

In [16]:
if UNDO_EDITRAW:    # Previously I changed exp_type of the raw files. I found better ways to do it
                    # This undoes that mistake
    for ii, thisdir in enumerate(dirs_to_handle) :   
        raw_files[ii] = glob.glob(thisdir+ '*nrs*_uncal.fits') #list the uncalibrated (level 1b) files.
        raw_files[ii] = sorted(raw_files[ii])
        counter = 0
        for exposure in raw_files[ii]:
            with fits.open(exposure, mode='update') as test:
                head = test[0].header
                if head["EXP_TYPE"] == 'NRS_LAMP' : raise Exception('ERROR! There should not be any EXP_TYPE NRS_LAMP frames!')
                if ('EXP_TYP0' in head):
                    #print("DEBUG, orig is", head['EXP_TYPE'], head['EXP_TYP0'],)
                    head['EXP_TYPE'] = head['EXP_TYP0']
                    del head['EXP_TYP0']
                    counter +=1
                #else: print('I am not sure what happened.  Debugging.  Here is EXP_TYPE, EXP_TYP0:', head["EXP_TYPE"], head["EXP_TYP0"], exposure)
                    
                if (('OPMODE' in head) and ('OPMODE0' in head)):
                    head['OPMODE'] = head['OPMODE0']  # Restore this
                    del head['OPMODE0']
                    counter += 1
                #test.writeto(exposure, overwrite=True)  # open as update takes care of this
        print(thisdir, "dir", ii, "of", len(dirs_to_handle) - 1, ", Checked", len(raw_files[ii]), "files, and changed ", counter, 'keywords')

In [17]:
SAVE_RATEINTS = False

if RUN_LEVEL1 :
    print('Starting Stage 1 reductions!')
    if pipev == '1.20.2':
        param_nested_dict = {'jump':{'expand_large_events': True}}
    elif ((pipev == '2.0.1') or (pipev == '3.0.0')):
        # We fit the ramp here.  For now, we do NOT clean 1/f noise here; we do that at Spec2
        param_nested_dict = {'jump': {'expand_large_events': True}, 'picture_frame': {'skip': False}, 'clean_flicker_noise': {'skip': True}}
    else : raise Exception("ERROR, do not have parameters for this pipeline version", pipev)

    for ii, thisdir in enumerate(dirs_to_handle):   
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        print("Reducing", input_path, "which is dir", ii+1, "of", len(dirs_to_handle))
        rawfiles_thisepoch = glob.glob(input_path + '*nrs*_uncal.fits') #list the uncalibrated (level 1b) files.
        output_dir = output_path + folder_L2a
        outptd = [output_dir for _ in range(len(rawfiles_thisepoch))]
        list_of_dicts = [param_nested_dict for _ in range(len(rawfiles_thisepoch))]
        #predownload_crds(rawfiles_thisepoch, context=mycontext)  # download any needed reference files BEFORE multiprocessing
        if (max_filesize(thisdir) > max_filesize_multiprocess): these_cores = 2   # multiprocess big files can run out of memory, crash computer hard
        else: these_cores = 8
        nfiles = len(rawfiles_thisepoch)
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S} — " f"Starting {nfiles} files with {these_cores} workers", flush=True)
        with Pool(these_cores, maxtasksperchild=10) as pool:
            results = pool.starmap(run_jwst_det1, zip(rawfiles_thisepoch, outptd, list_of_dicts, [SAVE_RATEINTS] * nfiles))
        failed = [name for name, result in zip(rawfiles_thisepoch, results) if result is None]
        if failed: 
            raise RuntimeError(f"Detector1 failed for: {failed}")

In [ ]:
if RUN_MASKS:
    mask_param_nested_dict = {'mask_MSA_by_wcs': True, 'mask_FS_by_wcs': True,
                              'mask_by_illumination': True, 'mask_illumination_nsigma': 5.0,
                              'mask_FS_strip': False, 'mask_FSprism': False, 'mask_grating_extrapolate': True}
    print('Making custom NIRSpec masks')
    for ii, thisdir in enumerate(dirs_to_handle):
        print('dir', ii, 'of ', len(dirs_to_handle))
        epochname, targname, input_path, output_path = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        files_to_mask = sorted(glob.glob(output_path + folder_L2a + '*nrs*_rate.fits'))
        if not files_to_mask:
            print('No files to mask in', output_path + folder_L2a)
            continue
        for raw_file in glob.glob(input_path + '*nrs*_uncal.fits'): 
            download_metadata(Path(raw_file))
        #predownload_crds(files_to_mask, context=mycontext) 
        these_cores = min(9, len(files_to_mask)) 
        with Pool(these_cores, maxtasksperchild=10) as pool: 
            masks = pool.starmap(make_rate_custom_mask, zip(files_to_mask, [input_path] * len(files_to_mask), [mask_param_nested_dict] * len(files_to_mask))) 
        failed = [name for name, result in zip(files_to_mask, masks) if result is None]
        if failed:
            raise RuntimeError(f"Mask generation failed for: {failed}")

In [18]:
if RUN_LEVEL2 :
    # Clean 1/f noise w clean flicker noise at Spec2 stage, using a custom mask file. 
    # clean flicker noise is better behaved here than when run w Detector1, where it can create crazy artifacts.  Though patches help
    # It's also much easier to fine-tune the 1/f noise parameters here, because this stage is much faster than Detector1
    # the custom run_jwst_spec2 function deals w the full-frame RATE file, then starts treating this like a fixed slit program
    print('Starting Stage 2 reductions!')
    if pipev == '1.20.2':
        raise RuntimeError("I dont have good parameters for this scenario")
    elif ((pipev == '2.0.1') or (pipev == '3.0.0')):
        param_nested_dict = {"clean_flicker_noise": {"skip": False, "autoparam": False, "fit_method": "fft", \
                "background_method": None, 'save_results':False, 'save_noise': False, 'save_background': False, "fit_by_channel": False}, "srctype": {"source_type": "EXTENDED"}}     
    else: raise Exception("ERROR, do not have parameters for this pipeline version", pipev)
    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        files_to_run = glob.glob(output_path + folder_L2a + '*nrs*_rate.fits')
        print('Reducing dir', output_path, 'which is dir', ii+1, 'of', len(dirs_to_handle))
        output_dir =  output_path + folder_L2b
        outptd = [output_dir for _ in range(len(files_to_run))]
        list_of_dicts = [param_nested_dict for _ in range(len(files_to_run))]
        #predownload_crds(files_to_run, context=mycontext)  # download any needed reference files BEFORE multiprocessing
        these_cores = 10
        with Pool(these_cores) as pool:
            results = pool.starmap(run_jwst_spec2, zip(files_to_run, outptd, list_of_dicts))
        failed = [name for name, result in zip(files_to_run, results) if not result] 
        if failed:  raise RuntimeError(f"Spec2 failed for: {failed}")

Starting Stage 2 reductions!
Reducing dir /Volumes/Rawdata/REDUCED_prism_extra/2023-02-14/VID02418041001/ which is dir 1 of 82


2026-11-24 08:45:24,184 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:24,324 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:24,545 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:25,797 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'


Reducing dir /Volumes/Rawdata/REDUCED_prism_extra/2023-02-14/VID02418027001/ which is dir 2 of 82


2026-11-24 08:45:36,820 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:36,871 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:36,871 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:37,043 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'


Reducing dir /Volumes/Rawdata/REDUCED_prism_extra/2023-02-22/VID02418002001/ which is dir 3 of 82


2026-11-24 08:45:47,268 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:47,292 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:47,365 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:45:47,403 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'


... (154 outputs left out)

Reducing dir /Volumes/Rawdata/REDUCED_prism_extra/2023-01-12/VID02418033001/ which is dir 81 of 82


2026-11-24 08:59:22,292 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:22,304 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:22,366 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:23,729 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'


Reducing dir /Volumes/Rawdata/REDUCED_prism_extra/2022-08-05/VID01373020001/ which is dir 82 of 82


2026-11-24 08:59:32,981 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:33,039 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:33,180 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-11-24 08:59:34,339 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'


In [19]:
if RUN_LEVEL3:  #   Level 3 processing for each epoch
    spec3_param_nested_dict = {'outlier_detection': {'skip': False}}
    jobs = []
    for ii, thisdir in enumerate(dirs_to_handle):  
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        L3_destripedir = output_path + folder_L3
        L3_extractdir  = output_path + folder_extracted
        for foo in (L3_destripedir, L3_extractdir):
            if os.path.exists(foo) == False:       # make a L3_destripe and L3_extract directory
                print('Creating folder ' + foo)
                os.system('mkdir ' + foo) # creates the folder
        # Write the associations needed to extract the spectra
        allmsa_calfiles  = glob.glob(output_path + folder_L2b + '/*cal.fits') 
        allmsa_asnfile   = L3_destripedir + 'L3asn.json'
        writel3asn(allmsa_calfiles, None, allmsa_asnfile, epochname + '_' + targname) 
        jobs.append((allmsa_asnfile, L3_destripedir, spec3_param_nested_dict))
    these_cores = 8
    with Pool(these_cores) as pool:
        pool.starmap(run_jwst_spec3, jobs)

Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2023-03-13/VID02418047001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2024-10-02/VID04645013001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2023-02-14/VID02418041001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2022-10-26/VID02418049001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2022-10-29/VID02418036001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2023-06-12/VID02418059001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2025-01-05/VID04645002001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2022-11-03/VID02418028001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2024-01-21/VID04028012001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2025-02-15/VID05940003001/L3_destripe/
Starting Spec3: /Volumes/Rawdata/REDUCED_prism_extra/2022-11-03/VID02418020001/L

In [20]:
if RUN_EXTRACT:
    clip_nan = 5
    jobs = []
    for ii, thisdir in enumerate(dirs_to_handle):
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        L3_destripedir = output_path + folder_L3
        L3_extractdir  = output_path + folder_extracted
        print(ii + 1 , "of", len(dirs_to_handle), L3_destripedir)
        jobs.append((L3_destripedir, L3_extractdir, clip_nan))
    these_cores = 8
    with Pool(these_cores) as pool:
        extract_reports = pool.starmap(run_extract, jobs) # Runs Jane's custom median combine for FS backgrounds
    summarize_extract(extract_reports)

1 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-02-14/VID02418041001/L3_destripe/
2 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-02-14/VID02418027001/L3_destripe/
3 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-02-22/VID02418002001/L3_destripe/
4 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2022-10-26/VID02418049001/L3_destripe/
5 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-01-05/VID02418015001/L3_destripe/
6 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-01-05/VID02418039001/L3_destripe/
7 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-06-12/VID02418059001/L3_destripe/
8 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2023-01-02/VID02418013001/L3_destripe/
9 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2024-11-21/VID05940001001/L3_destripe/
10 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2022-10-29/VID02418036001/L3_destripe/
11 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2024-01-21/VID04028012001/L3_destripe/
12 of 82 /Volumes/Rawdata/REDUCED_prism_extra/2022-09-13/VID01272008001/L3

## Now, go see MSA_FixedSlit_analysis_plotting.ipynb, to plot results